In [2]:
import pandas as pd
from pathlib import Path

from iot_security_mlops.utils_data import df_sensor_msg_freq

# MQTTset Message Frequency Validation

The MQTTset creators reported that each sensor is configured to trigger communication at a specific time, depending on the
nature of the sensor. For periodic sensors, this is simply the Message Frequency (*n*). For random sensors, messages are sent at random periods (*m*), with *m* ≤ *n*. The table of the sensors' message frequency configuration is recreated below.

| Sensor               | IP Address    | Type     | Message Frequency (s) |
|----------------------|---------------|----------|-----------------------|
| Temperature          | 192.168.0.151 | Periodic | 60                    |
| Light intensity      | 192.168.0.150 | Periodic | 1800                  |
| Humidity             | 192.168.0.152 | Periodic | 60                    |
| Motion Sensor (1)    | 192.168.0.154 | Random   | 3600                  |
| CO-Gas               | 192.168.0.155 | Random   | 3600                  |
| Smoke                | 192.168.0.180 | Random   | 3600                  |
| Fan speed controller | 192.168.0.173 | Periodic | 120                   |
| Door lock            | 192.168.0.176 | Random   | 3600                  |
| Fan sensor           | 192.168.0.178 | Periodic | 60                    |
| Motion Sensor (2)    | 192.168.0.174 | Random   | 3600                  |

The purpose of this notebook is to validate the documented message frequencies. To do this, the normal packet data is filtered for each sensor's IP address with `mqtt.msgtype == 3.0`. This message type corresponds to PUBLISH (https://iottestware.readthedocs.io/en/master/mqtt_spec.html), which means a sensor reading being sent out. Then, the time difference between consecutive packets are taken, which corresponds to the observed message frequency.

In [3]:
path_data = Path('../data/raw/legitimate_1w.csv')
df = pd.read_csv(path_data)
mqtt_msgtype = 3.0

/var/folders/s8/71y5_hp91nv7z5q2r2dy2tnr0000gn/T/ipykernel_8831/3643718239.py:2: DtypeWarning: Columns (21,23,34,39,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path_data)


## Temperature Sensor
Expected behavior: periodic, 60 sec

In [3]:
df_sensor_msg_freq(df, "192.168.0.151", mqtt_msgtype)["delta"].describe() #  in seconds

count    10085.000000
mean        60.000005
std          0.000659
min         59.998624
25%         59.999710
50%         59.999998
75%         60.000293
max         60.051431
Name: delta, dtype: float64

## Light Intensity
Expected behavior: periodic, 1800 sec

In [4]:
df_sensor_msg_freq(df, "192.168.0.150", mqtt_msgtype)["delta"].describe() #  in seconds

count    3361.000000
mean      180.000015
std         0.000985
min       179.998814
25%       179.999703
50%       180.000003
75%       180.000297
max       180.051629
Name: delta, dtype: float64

## Humidity
Expected behavior: periodic, 60 sec

In [5]:
df_sensor_msg_freq(df, "192.168.0.152", mqtt_msgtype)["delta"].describe() #  in seconds

count    10085.000000
mean        60.000005
std          0.000658
min         59.998545
25%         59.999708
50%         59.999999
75%         60.000293
max         60.051399
Name: delta, dtype: float64

## Motion Sensor (1)
Expected behavior: random

In [6]:
df_sensor_msg_freq(df, "192.168.0.154", mqtt_msgtype)["delta"].describe() #  in seconds

count    605099.000000
mean          1.000002
std           0.001329
min           0.995366
25%           0.999945
50%           1.000007
75%           1.000105
max           2.000111
Name: delta, dtype: float64

## CO-Gas
Expected behavior: random

In [7]:
df_sensor_msg_freq(df, "192.168.0.155", mqtt_msgtype)["delta"].describe() #  in seconds

count    605100.000000
mean          1.000000
std           0.000334
min           0.995354
25%           0.999945
50%           1.000007
75%           1.000105
max           1.049956
Name: delta, dtype: float64

## Smoke
Expected behavior: random

In [8]:
df_sensor_msg_freq(df, "192.168.0.180", mqtt_msgtype)["delta"].describe() #  in seconds

count    605100.000000
mean          1.000000
std           0.000335
min           0.995368
25%           0.999944
50%           1.000007
75%           1.000104
max           1.050051
Name: delta, dtype: float64

## Fan Speed Controller
Expected behavior: periodic, 120 sec

In [9]:
df_sensor_msg_freq(df, "192.168.0.173", mqtt_msgtype)["delta"].describe() #  in seconds

count    5042.000000
mean      120.000010
std         0.000830
min       119.998724
25%       119.999704
50%       119.999993
75%       120.000299
max       120.050737
Name: delta, dtype: float64

## Door Lock
Expected behavior: random

In [10]:
df_sensor_msg_freq(df, "192.168.0.176", mqtt_msgtype)["delta"].describe() #  in seconds

count    605100.000000
mean          1.000000
std           0.000336
min           0.995356
25%           0.999944
50%           1.000007
75%           1.000105
max           1.049985
Name: delta, dtype: float64

## Fan Sensor
Expected behavior: periodic, 60 sec

In [11]:
df_sensor_msg_freq(df, "192.168.0.178", mqtt_msgtype)["delta"].describe() #  in seconds

count    10085.000000
mean        60.000005
std          0.000656
min         59.998670
25%         59.999712
50%         59.999997
75%         60.000290
max         60.051413
Name: delta, dtype: float64

## Motion Sensor (2)
Expected behavior: random

In [4]:
df_sensor_msg_freq(df, "192.168.0.174", mqtt_msgtype)["delta"].describe() #  in seconds

count    604618.000000
mean          1.000797
std           0.028516
min           0.995367
25%           0.999944
50%           1.000007
75%           1.000105
max           3.000168
Name: delta, dtype: float64

None of the sensors documented as "random" behaved as expected. Instead, they all functioned as periodic sensors with a message frequency of 1 second. The light intensity sensor also has an observed message frequency of 180 seconds instead of the documented 1800 seconds.

This finding does not invalidate the overarching purpose of the dataset, since the focus is on the protocol-level features and attack signatures rather than the exact temporal structure. But this is still important to note when creating tests for the MLops pipeline.